# Regression

## General

In [1]:
import pandas as pd
import math
import quandl
quandl.ApiConfig.api_key="yHJfykCCxpokgr4q4hhN"

In [2]:
df= quandl.get('EURONEXT/ADYEN')

In [3]:
print(df.head())

             Open    High    Low   Last     Volume     Turnover
Date                                                           
2018-06-13  400.0  503.90  400.0  455.0  1529232.0  674779793.0
2018-06-14  469.0  484.00  438.0  438.0   148388.0   68540008.0
2018-06-15  435.4  437.25  415.1  420.0   116467.0   49560634.0
2018-06-18  422.0  422.75  401.2  411.0    88873.0   36471156.0
2018-06-19  409.7  425.00  401.9  412.5    63138.0   26221340.0


In [4]:
df= df[['Open', 'High', 'Low', 'Last','Volume']]
df.columns=['Open', 'High', 'Low', 'Close','Volume']

In [5]:
df['high minus low percent']=(df['High']-df['Low'])/df['Low']

In [6]:
df['percent change']=(df['Close']-df['Open'])/df['Open']

In [7]:
df=df[['Close','high minus low percent', 'percent change','Volume']]

In [8]:
print(df.head())

            Close  high minus low percent  percent change     Volume
Date                                                                
2018-06-13  455.0                0.259750        0.137500  1529232.0
2018-06-14  438.0                0.105023       -0.066098   148388.0
2018-06-15  420.0                0.053361       -0.035370   116467.0
2018-06-18  411.0                0.053714       -0.026066    88873.0
2018-06-19  412.5                0.057477        0.006834    63138.0


In [9]:
## Defining a label

In [10]:
forecast_col='Close' ##rajouter une colonne forecast à data frame

In [11]:
df.fillna(-9999, inplace=True) ##replacing empty data with -9999

In [12]:
forecast_out=int(math.ceil(0.1*len(df))) ##partie entière de 1% du nombre de données dans df

In [13]:
df['label']=df[forecast_col].shift(-forecast_out) ##décaler les cellules de forecastout vers le haut, elles vont maintenant correspondre à d'autres valeurs d'écart et de volume
df_2=df.copy()
df.dropna(inplace=True)

In [14]:
import numpy as np
from sklearn import preprocessing ##for scaling the data between -1 and 1 to improve caclculus speed
from sklearn import model_selection, svm ##scaling vector method
from sklearn.linear_model import LinearRegression

In [15]:
X=np.array(df.drop(['label'],1)) ## 1 is to specify the axis, otherwise you get a "axis not defined" error
print (X[0:5])

[[ 4.55000000e+02  2.59750000e-01  1.37500000e-01  1.52923200e+06]
 [ 4.38000000e+02  1.05022831e-01 -6.60980810e-02  1.48388000e+05]
 [ 4.20000000e+02  5.33606360e-02 -3.53697749e-02  1.16467000e+05]
 [ 4.11000000e+02  5.37138584e-02 -2.60663507e-02  8.88730000e+04]
 [ 4.12500000e+02  5.74769843e-02  6.83426898e-03  6.31380000e+04]]


In [16]:
Y=np.array(df['label'])

In [17]:
X=preprocessing.scale(X)

In [18]:
print(len(X),len(Y))

589 589


In [19]:
X_train, X_test, Y_train, Y_test=model_selection.train_test_split(X, Y, test_size=0.2) ##mélanger les données pour en sélectionner 80% d'entraînement et 20% de test

In [20]:
clf=LinearRegression() ##classifier
clf.fit(X_train, Y_train) ##train it
accuracy = clf.score(X_test, Y_test) ## test it

In [21]:
print(accuracy) ##accuracy is the squared error

0.7942135077375325


In [22]:
print (forecast_out)

66


In [23]:
clc=svm.SVR() ##classifier
clc.fit(X_train, Y_train) ##train it
accuracy = clc.score(X_test, Y_test) ## test it
print(accuracy)

-0.08423176673211219


In [24]:
clc_1=svm.SVR(kernel='poly') ##classifier en régression polynomiale plutôt que linéaire
clc_1.fit(X_train, Y_train) ##train it
accuracy = clc_1.score(X_test, Y_test) ## test it
print(accuracy)

-0.0795887071529906


In [25]:
clf=LinearRegression(n_jobs=10) ##classifier en traitant les données d'entraînement 10 par 10: ça va plus vite
clf.fit(X_train, Y_train) ##train it
accuracy = clf.score(X_test, Y_test) ## test it
print(accuracy)

0.7942135077375325


In [28]:
X=np.array(df_2.drop(['label'],1))
X=preprocessing.scale(X)
X_lately=X[-forecast_out:] ## because there is no label for the lowest data, which is the most recent
X=X[:-forecast_out]

df_2.dropna(inplace=True)
Y=np.array(df_2['label'])[:-forecast_out]
print (len(X), len(Y))

X_train, X_test, Y_train, Y_test=model_selection.train_test_split(X, Y, test_size=0.2) ##mélanger les données pour en sélectionner 80% d'entraînement et 20% de test
clf=LinearRegression(n_jobs=1) ##classifier en traitant les données d'entraînement 10 par 10: ça va plus vite
clf.fit(X_train, Y_train) ##train it
accuracy = clf.score(X_test, Y_test) ## test it
print(accuracy)

523 523
0.6354443592117194


In [ ]:
forecast_set=clf.predict(X_prediction)
print (forecast_set)

In [ ]:
import datetime
from datetime import timedelta
import matplotlib.pyplot as plt
from matplotlib import style
style.use('ggplot')

df_2['Forecast']=np.nan
last_date=df_2.iloc[-1].name ##trouver la date la plus récente du dataframe
print(last_date)
##last_unix=last_date.timestamp() ##la convertir
##one_day=86400
##next_unix=last_unix+one_day ##lui ajouter un jour dans les bonnes unités
one_day=datetime.timedelta(days=1)
print (one_day)
next_day=last_date+one_day
print(next_day)

for i in forecast_set: ##définir les dates ultérieures qui seront en abscisse
    next_date=next_day
    next_day+=one_day
    df_2.loc[next_date]=[np.nan for j in range (len(df_2.columns)-1)]+[i] #adding lines to the end of the dataframe indexed by next_date. Since the Forecast column will be empty aprt from the predicted date, we can plot them both together
df_2['Close'].plot()
df_2['Forecast'].plot()
plt.legend(loc=4) ##choosing where to print the legend
plt.x_label('Date')
plt.y_label=('Price')
plt.show()

## Picking and scaling

In [ ]:
import pickle ## allows you to save the classifier once it is trained so that you don't have to train it again
clf=LinearRegression(n_jobs=-1)
clf.fit(X_train, Y_train)
with open('linearregression.pickle','wb') as f:
    pickle.dump(clf,f)   ##open the file with the intention of writing in it (f)
pickle_in=open('linearregression.pickle', 'rb')
clf=pickle.load(pickle_in) ##saved in the current folder

## Writing our own linear regression

In [ ]:
from statistics import mean,variance
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import style
style.use('fivethirtyeight')

xs=np.array([1, 2, 3, 4, 5, 6], dtype=np.float64)
ys=np.array([5,4,6,5,6,7], dtype=np.float64)

def bestFitSlope(xs,ys):
    num=mean(xs)*mean(ys)-mean(np.multiply(xs,ys))
    den=-mean(xs*xs)+mean(xs)**2
    return num/den

def ordonnéeAOrigine(m, xs,ys):
    return mean(ys)-m*mean(xs)

m=bestFitSlope(xs,ys)
b= ordonnéeAOrigine(m, xs, ys)
print(m, b)

regression_line=[m*x+b for x in xs]

plt.scatter(ys,xs)
plt.plot(regression_line,xs)
plt.show()

In [ ]:
def Rsquared(xs, ys, m, b): ##coefficient of determination should be as close to one as possible 
    moyenne=mean(ys)
    modelError=0
    intrinsicError=0
    for i in range(len(ys)):
        modelError+=(ys[i]-m*xs[i]-b)**2
        intrinsicError+=(ys[i]-moyenne)**2
    return 1-modelError/intrinsicError
print(Rsquared(xs, ys, m, b))  

In [ ]:
import random
def createDataset (howmany, variability, step=2, correlation=False):
    value=1
    ys=[]
    for i in range (howmany):
        y=value+ random.randrange(-variability, variability)
        ys.append(y)
        if correlation and correlation=="pos":
            value+=step
        elif correlation and correlation=="neg":
            value-=step
    xs=[i for i in range (howmany)]
    return np.array(xs, dtype=np.float64), np.array(ys, dtype=np.float64)

def predict_y (m, b, x):
    return m*x+b

predict_x=8

xs, ys=createDataset(40, 10, 2, "neg")

m=bestFitSlope(xs,ys)
b= abscisseAOrigine(m, xs, ys)
print(m, b)

regression_line=[m*x+b for x in xs]

plt.scatter(ys,xs)
plt.plot(regression_line,xs)
plt.show()

Rsquared(xs,ys, m, b)